In [1]:
import json
from collections import Counter
from pathlib import Path

In [2]:
vehicle_map_path = Path(
    "/home/chen/workspace/dcd-ctrlsim/scenario_json/vehicle_map_1k_train_no_offroad.json"
)
with vehicle_map_path.open("r", encoding="utf-8") as f:
    vehicle_map = json.load(f)

In [3]:
veh_number_counts = Counter(
    scene_info["opponent_vehicle_num"] for scene_info in vehicle_map.values()
)

print(f"Total scenarios: {sum(veh_number_counts.values())}")
print("veh_number\tscenario_count")
for veh_number, scenario_count in sorted(veh_number_counts.items()):
    print(f"{veh_number}\t{scenario_count}")

Total scenarios: 1000
veh_number	scenario_count
0	43
1	208
2	184
3	129
4	102
5	66
6	65
7	203


In [4]:
missing_ego_vehicle_id_count = sum(
    scene_info.get("ego_vehicle_id") is None
    or scene_info.get("ego_vehicle_id") == ""
    or scene_info.get("ego_vehicle_id") == -1
    for scene_info in vehicle_map.values()
)
total_scenarios = len(vehicle_map)
missing_ratio = missing_ego_vehicle_id_count / total_scenarios

print(f"Total scenarios: {total_scenarios}")
print(f"Missing ego_vehicle_id scenarios: {missing_ego_vehicle_id_count}")
print(f"Missing ratio: {missing_ratio:.4%}")


Total scenarios: 1000
Missing ego_vehicle_id scenarios: 0
Missing ratio: 0.0000%


In [5]:
import json
import math
import sys
from pathlib import Path

from tqdm.auto import tqdm

project_root = Path("/home/chen/workspace/dcd-ctrlsim")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from envs.nocturne_ctrlsim.adversarial import NocturneCtrlSimAdversarial
from envs.nocturne_ctrlsim.core.level import build_zero_tilt_level

log_dir = Path(
    "/media/chen/Dataset/logs/dcd/steps4096000-proc16-roll256-plr1-edit1-tiltper_vehicle-kl1-prw0_0"
)
meta_path = log_dir / "meta.json"
with meta_path.open("r", encoding="utf-8") as f:
    meta = json.load(f)
args = meta.get("args", meta)

scenario_index_path = project_root / "scenario_json" / "scenario_index_10k_train.json"
with scenario_index_path.open("r", encoding="utf-8") as f:
    scenario_index = json.load(f)
scenario_ids = scenario_index["scenario_ids"]

max_static_speed = 0.1
max_ego_path_distance = 2.0


def point_segment_distance(
    px: float,
    py: float,
    ax: float,
    ay: float,
    bx: float,
    by: float,
) -> float:
    """Return the shortest distance from a point to a line segment."""
    abx = bx - ax
    aby = by - ay
    apx = px - ax
    apy = py - ay
    ab_sq = abx * abx + aby * aby
    if ab_sq == 0.0:
        return math.hypot(px - ax, py - ay)

    t = max(0.0, min(1.0, (apx * abx + apy * aby) / ab_sq))
    qx = ax + t * abx
    qy = ay + t * aby
    return math.hypot(px - qx, py - qy)


def distance_to_polyline_points(px: float, py: float, points: list[tuple[float, float]]) -> float:
    """Return the shortest distance from a point to one polyline."""
    if not points:
        return float("inf")
    if len(points) == 1:
        point_x, point_y = points[0]
        return math.hypot(px - point_x, py - point_y)

    best_distance = float("inf")
    for (start_x, start_y), (end_x, end_y) in zip(points, points[1:]):
        distance = point_segment_distance(
            px,
            py,
            start_x,
            start_y,
            end_x,
            end_y,
        )
        if distance < best_distance:
            best_distance = distance
    return best_distance


def get_vehicle_gt_path_points(env, veh_id: int) -> list[tuple[float, float]]:
    """Return valid GT trajectory points for one runtime-controlled vehicle."""
    gt_data = env._gt_data_dict.get(veh_id)
    if gt_data is None:
        return []

    path_points = []
    for pos_x, pos_y, _heading, _speed, exists, *_rest in gt_data["traj"]:
        if not bool(exists):
            continue
        path_points.append((float(pos_x), float(pos_y)))
    return path_points


def get_controlled_gt_path_points(env) -> list[list[tuple[float, float]]]:
    """Return GT paths for ego and all opponent vehicles."""
    controlled_vehicle_ids = list(env.opponent_vehicle_ids)
    if env.ego_vehicle is not None:
        controlled_vehicle_ids.insert(0, env.ego_vehicle.getID())

    controlled_paths = []
    for veh_id in controlled_vehicle_ids:
        path_points = get_vehicle_gt_path_points(env, veh_id)
        if path_points:
            controlled_paths.append(path_points)
    return controlled_paths


def get_vehicle_xy(veh) -> tuple[float, float]:
    """Return the runtime vehicle position as x/y floats."""
    position = veh.getPosition()
    return float(position.x), float(position.y)


def get_runtime_blocking_vehicle_ids(env) -> list[int]:
    """Return static runtime vehicles that sit on any controlled GT route."""
    controlled_gt_paths = get_controlled_gt_path_points(env)
    if not controlled_gt_paths:
        return []

    protected_ids = set(env.opponent_vehicle_ids)
    if env.ego_vehicle is not None:
        protected_ids.add(env.ego_vehicle.getID())

    blocking_vehicle_ids = []
    for veh in env.vehicles:
        veh_id = veh.getID()
        if veh_id in protected_ids:
            continue

        px, py = get_vehicle_xy(veh)
        speed = float(veh.speed)
        if speed > max_static_speed:
            continue

        min_path_distance = min(
            distance_to_polyline_points(px, py, gt_path_points)
            for gt_path_points in controlled_gt_paths
        )
        if min_path_distance <= max_ego_path_distance:
            blocking_vehicle_ids.append(veh_id)

    return blocking_vehicle_ids


env = NocturneCtrlSimAdversarial(
    scenario_index_path=args["scenario_index_path"],
    opponent_checkpoint=args["opponent_checkpoint"],
    scenario_data_dir=args["scenario_data_dir"],
    preprocess_dir=args["preprocess_dir"],
    vehicle_map_path=args["vehicle_map_path"],
    device="cpu",
    tilting_mode=args["tilting_mode"],
    mutation_mode=args["mutation_mode"],
    student_accel_discretization=args["student_accel_discretization"],
    student_steer_discretization=args["student_steer_discretization"],
    obs_dim=None,
    remove_background_vehicles=args["remove_background_vehicles"],
    show_vehicle_ids=args["show_vehicle_ids"],
    show_tilting_params=args["show_tilting_params"],
    show_ego_vehicle_selection=args["show_ego_vehicle_selection"],
    opponent_runtime_mode=args.get("opponent_runtime_mode") or "normal",
    inference_precision=args.get("inference_precision", "fp32"),
)

flagged_scenario_ids = []
flagged_vehicle_count = 0

for scenario_id in tqdm(scenario_ids, desc="Scanning runtime scenarios"):
    level = build_zero_tilt_level(
        scenario_id=scenario_id,
        seed=1,
        tilting_mode=env.tilting_mode,
        per_vehicle_tilting_length=env.per_vehicle_tilting_length,
    )
    env.reset_to_level(level)
    blocking_vehicle_ids = get_runtime_blocking_vehicle_ids(env)
    if blocking_vehicle_ids:
        flagged_scenario_ids.append(scenario_id)
        flagged_vehicle_count += len(blocking_vehicle_ids)

scanned_scenarios = len(scenario_ids)
flagged_scenario_count = len(flagged_scenario_ids)
flagged_ratio = flagged_scenario_count / scanned_scenarios if scanned_scenarios else 0.0

print(f"Scenario index: {scenario_index_path}")
print(f"Runtime scanned scenarios: {scanned_scenarios}")
print(f"Scenarios with GT-route blocking vehicles: {flagged_scenario_count}")
print(f"Scenario ratio: {flagged_ratio:.4%}")
print(f"GT-route blocking vehicle count: {flagged_vehicle_count}")
print(f"Flagged scenarios: {flagged_scenario_ids}")


/home/chen/miniconda3/envs/dcd-ctrlsim/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/chen/miniconda3/envs/dcd-ctrlsim/lib/python3.10/site-packages/gym/envs/registration.py:307: DeprecationWarning: The package name gym_minigrid has been deprecated in favor of minigrid. Please uninstall gym_minigrid and install minigrid with `pip install minigrid`. Future releases will be maintained under the new package name minigrid.
  fn()
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of

KeyboardInterrupt: 

  File "/home/chen/miniconda3/envs/dcd-ctrlsim/lib/python3.10/concurrent/futures/process.py", line 240, in _process_worker
    call_item = call_queue.get(block=True)
  File "/home/chen/miniconda3/envs/dcd-ctrlsim/lib/python3.10/concurrent/futures/process.py", line 240, in _process_worker
    call_item = call_queue.get(block=True)
  File "/home/chen/miniconda3/envs/dcd-ctrlsim/lib/python3.10/concurrent/futures/process.py", line 240, in _process_worker
    call_item = call_queue.get(block=True)
  File "/home/chen/miniconda3/envs/dcd-ctrlsim/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/chen/miniconda3/envs/dcd-ctrlsim/lib/python3.10/multiprocessing/queues.py", line 102, in get
    with self._rlock:
  File "/home/chen/miniconda3/envs/dcd-ctrlsim/lib/python3.10/multiprocessing/queues.py", line 102, in get
    with self._rlock:
  File "/home/chen/miniconda3/envs/dcd-ctrlsim/lib/python3.10/concurrent/futures/process.py", line 240, in _proces

  File "/home/chen/miniconda3/envs/dcd-ctrlsim/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/chen/miniconda3/envs/dcd-ctrlsim/lib/python3.10/multiprocessing/synchronize.py", line 95, in __enter__
    return self._semlock.__enter__()
  File "/home/chen/miniconda3/envs/dcd-ctrlsim/lib/python3.10/multiprocessing/queues.py", line 102, in get
    with self._rlock:
  File "/home/chen/miniconda3/envs/dcd-ctrlsim/lib/python3.10/multiprocessing/synchronize.py", line 95, in __enter__
    return self._semlock.__enter__()
  File "/home/chen/miniconda3/envs/dcd-ctrlsim/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/chen/miniconda3/envs/dcd-ctrlsim/lib/python3.10/multiprocessing/queues.py", line 102, in get
    with self._rlock:
  File "/home/chen/miniconda3/envs/dcd-ctrlsim/lib/python3.10/multiprocessing/synchronize.py", line 95, in __enter__
    retur